# Decision Trees: Bias, Variance and Stopping Rules

**DSAI 621 — Week 6.** A regression tree cuts the predictor space into boxes and predicts the
average response in each box (ISL 8.1). How big the tree is decides almost everything: a tree with
two leaves can't follow the data, and a tree with a leaf for every observation follows the noise. In
this notebook you will:

1. fit a few trees to one training set, and read them;
2. measure the **bias** and **variance** of a tree by simulation, using many training sets from the
   same population;
3. see how four ways of limiting a tree (`max_depth`, `min_samples_leaf`, `max_leaf_nodes` and
   cost-complexity pruning, ISL 8.4) move it along the bias–variance trade-off.

All of the code is written for you: run each cell, read what it does, and change it to try
things out. Short exercises marked **E1**, **E2**, … ask you to stop and look at what you just
made, and answer in words. Exercises marked **T1**, **T2**, … ask you to change a number in a cell,
rerun it, and explain what changed. Each one says which cells to rerun and whether to change the
number back afterwards.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor, export_text

# random number generator for reproducibility
rng = np.random.default_rng(621)

### The data and the helper functions

We use Example B from the splines lab and the lecture notes, so the trees can be compared with the
splines you fitted there. One predictor $x$, uniform on $[0, 1]$, and
$$y = f(x) + \varepsilon, \qquad f(x) = \frac{\sin(12(x + 0.2))}{x + 0.2}, \qquad \varepsilon \sim N(0, 1).$$
$f$ wiggles fast and with a large amplitude near $x = 0$ and flattens out towards $x = 1$. Each
training set has $n = 100$ observations.

Because the data are simulated, we can draw as many training sets as we like. For a way of fitting a
tree, `simulate` draws $B = 200$ training sets, fits a tree to each, and stores each tree's
predictions on a fine grid of $x$ values. From those, `bias_variance` computes, averaged over $x$:

- **squared bias**: how far the *average* tree (over training sets) is from $f$;
- **variance**: how much the trees vary from one training set to the next;
- **expected test MSE** $= \sigma^2 + \text{bias}^2 + \text{variance}$, the bias–variance
  decomposition from lecture 1, with $\sigma^2 = 1$ the irreducible error.

Every call to `simulate` uses the same seed, so every setting is fitted to **the same 200 training
sets**, and the differences between settings are not due to luck in the data. The argument `fit` is
a function that takes `X, y` and returns a fitted tree, such as
`lambda X, y: DecisionTreeRegressor(max_depth=3).fit(X, y)`.

In [ ]:
SIGMA = 1.0     # noise standard deviation
N = 100         # training set size
B = 200         # number of simulated training sets

# True function
def f_true(x):
    return np.sin(12 * (x + 0.2)) / (x + 0.2)

# Function to simulate data from true function with Gaussian noise
def make_data(n, rng, sigma=SIGMA):
    "A training set: X is n x 1 (sklearn wants a matrix), y has length n."
    x = rng.uniform(0, 1, n)
    y = f_true(x) + rng.normal(0, sigma, n)
    return x.reshape(-1, 1), y

x_grid = np.linspace(0, 1, 501).reshape(-1, 1)     # where we compare the trees with f

# Function to simulate B training sets, fit a tree to each, and return predictions on x_grid
def simulate(fit, B=B, n=N, sigma=SIGMA, seed=1):
    "Fit a tree to each of B training sets. Returns predictions on x_grid (B x 501), leaves, training MSEs."
    rng_sim = np.random.default_rng(seed)          # same seed: the same B training sets every time
    preds, leaves, train_mse = np.empty((B, len(x_grid))), np.empty(B), np.empty(B)
    for b in range(B):
        X, y = make_data(n, rng_sim, sigma)
        tree = fit(X, y)
        preds[b] = tree.predict(x_grid)
        leaves[b] = tree.get_n_leaves()
        train_mse[b] = np.mean((tree.predict(X) - y) ** 2)
    return preds, leaves, train_mse

# Function to compute bias^2, variance, and expected test MSE
def bias_variance(preds, sigma=SIGMA):
    "Squared bias, variance and expected test MSE, each averaged over x in [0, 1]."
    f = f_true(x_grid[:, 0])
    bias2 = np.mean((preds.mean(axis=0) - f) ** 2)
    var = np.mean(preds.var(axis=0))
    return bias2, var, sigma**2 + bias2 + var

# Function to summarize the results of a simulation
def summarize(fit, label, **kwargs):
    "One row of a results table for one way of fitting a tree."
    preds, leaves, train_mse = simulate(fit, **kwargs)
    bias2, var, test = bias_variance(preds)
    return {"setting": label, "mean leaves": leaves.mean(), "training MSE": train_mse.mean(),
            "bias^2": bias2, "variance": var, "expected test MSE": test}

# Function to plot the bias-variance tradeoff
def plot_tradeoff(df, title, xlabel="mean number of leaves"):
    "Squared bias, variance and expected test MSE against the average tree size."
    fig, ax = plt.subplots(figsize=(7, 4))
    x = df["mean leaves"]
    ax.plot(x, df["bias^2"], "o--", label="bias$^2$")
    ax.plot(x, df["variance"], "s-.", label="variance")
    ax.plot(x, df["expected test MSE"] - SIGMA**2, "^-", lw=2, label="expected test MSE $- \\sigma^2$")
    ax.plot(x, df["training MSE"], "x:", label="training MSE")
    ax.set(xscale="log", xlabel=xlabel, ylabel="value", title=title)
    ax.legend()
    plt.show()

# Simulate some training data
X, y = make_data(N, rng)
print("training data:", X.shape)

---
## 1. A few trees on one training set

The cell below fits trees of depth 1, 2, 3 and 6 to one training set. A tree of depth $d$ asks at
most $d$ questions on the way from the root to a leaf, so it has at most $2^d$ leaves. Each panel
shows the data, the true $f$ (dotted) and the tree's fit, which is a step function: one flat step
per leaf. The depth-2 tree is printed as text: `x <= 0.25` means the left branch is $x \le 0.25$
(sklearn writes $\le$ where ISL writes $<$; with cutpoints halfway between observations they mean the
same thing).

In [ ]:
depths = [1, 2, 3, 6]

# Plot the fitted trees for different depths
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
for ax, d in zip(axes, depths):
    tree = DecisionTreeRegressor(max_depth=d).fit(X, y)
    ax.scatter(X[:, 0], y, s=12, color="grey", alpha=0.7, label="data")
    ax.plot(x_grid[:, 0], f_true(x_grid[:, 0]), "k:", lw=2, label="true f")
    ax.plot(x_grid[:, 0], tree.predict(x_grid), lw=2, label="tree")
    ax.set(title=f"max_depth = {d}  ({tree.get_n_leaves()} leaves)", xlabel="x")
axes[0].set_ylabel("y")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(export_text(DecisionTreeRegressor(max_depth=2).fit(X, y), feature_names=["x"]))

**E1.** Read the depth-2 tree. Where is the first cut, and what does the tree predict on each side of
it? Why is the first cut there? (Think about the by-eye rule from class: where is the biggest jump
between two groups that are each fairly flat?)

<details>
<summary>Show answer: E1</summary>

**E1. Answer.** The first cut is at $x = 0.29$ (`x <= 0.29`). The left side is cut again at $x = 0.05$, and
the tree predicts 2.47 for $x \le 0.05$ and $-2.19$ for $0.05 < x \le 0.29$. The right side is cut at
$x = 0.60$, with predictions 1.04 for $0.29 < x \le 0.60$ and $-0.17$ beyond. The first cut is at 0.29
because that is where the biggest jump is between two large groups: to its left most points are in the
deep trough of $f$ (around $-2$), and to its right they are around 0 to 1.5. The split that reduces RSS
most is the one with the biggest gap between the two means, weighted by how many points are on each side,
which is the by-eye rule made precise.

</details>

**E2.** As the depth grows from 1 to 6, where do the new cuts go: into the left part of $[0, 1]$, where
$f$ wiggles, or the right part, where it is flat? Why does the greedy algorithm put them there? Does
the depth-6 tree look better or worse than the depth-3 tree, and where?

<details>
<summary>Show answer: E2</summary>

**E2. Answer.** Most of the new cuts go into the left part, where $f$ changes fast: three of the depth-3 tree's seven
cuts are below $x = 0.1$ (at 0.02, 0.05 and 0.08), following the steep drop from 3.4 to $-2.5$, and only
two are to the right of 0.5. The greedy algorithm makes the split that reduces RSS the most, and a cut reduces RSS most where the
mean changes most from one side to the other, which is where $f$ is steep. The depth-6 tree (39 leaves)
looks worse. It follows individual noisy points, such as the drop to about $-5$ near $x = 0.22$ and the
spike to 3.7 near $x = 0.47$. The depth-3 tree has visible steps, but it follows the shape of $f$
without chasing the noise.

</details>

---
## 2. Bias and variance of a tree

One training set can't tell us whether a fit is good in general, because a different training set
would give a different tree. The cell below fits a shallow tree (depth 2) and a **fully grown** tree
(no stopping rule at all: it splits until every leaf has one observation) to each of the 200 training
sets. For each, it plots 20 of the 200 fits (thin lines), the average of all 200 (thick line) and the
true $f$ (dotted), and prints the squared bias, variance and expected test MSE.

In [ ]:
# Settings for different tree configurations
settings = {"max_depth = 2": lambda X, y: DecisionTreeRegressor(max_depth=2).fit(X, y),
            "fully grown": lambda X, y: DecisionTreeRegressor().fit(X, y)}

# Plot truth and predictions for each setting, and summarize the results
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
rows = []
for ax, (label, fit) in zip(axes, settings.items()):
    preds, leaves, train_mse = simulate(fit)
    for b in range(20):
        ax.plot(x_grid[:, 0], preds[b], color="tab:blue", lw=0.6, alpha=0.4)
    ax.plot(x_grid[:, 0], preds.mean(axis=0), color="tab:red", lw=2.5, label="average of 200 trees")
    ax.plot(x_grid[:, 0], f_true(x_grid[:, 0]), "k:", lw=2, label="true f")
    ax.set(title=f"{label}  (mean leaves {leaves.mean():.0f})", xlabel="x", ylim=(-4, 6))
    bias2, var, test = bias_variance(preds)
    rows.append({"setting": label, "mean leaves": leaves.mean(), "training MSE": train_mse.mean(),
                 "bias^2": bias2, "variance": var, "expected test MSE": test})
axes[0].set_ylabel("fitted value")
axes[0].legend()
plt.tight_layout()
plt.show()
display(pd.DataFrame(rows).set_index("setting").round(3))

**E3.** Describe each panel. For which tree is the *average* fit far from $f$, and where? For which do
the 20 individual fits disagree most with each other? Match each description to the ${bias}^2$ and
variance columns of the table.

<details>
<summary>Show answer: E3</summary>

**E3. Answer.** For the depth-2 tree (4 leaves), the *average* fit is far from $f$: it is flat at about 0.8 where $f$
rises to 1.5 near $x = 0.45$, it misses the trough near $x = 0.7$ and the rise towards $x = 1$
completely, and it cuts off the peak at $x = 0$. That is the squared bias of 0.333. Its individual fits
lie close together, except near the cuts, whose position moves from one training set to the next. The
fully grown tree is the opposite: its average lies almost exactly on $f$ (squared bias 0.007), but the 20
individual fits are scattered widely around it, which is a variance of 1.007. The depth-2 tree's variance,
0.373, is not small either: four leaf means plus three cutpoints, all estimated from 100 noisy points.

</details>

**E4.** The fully grown tree has training MSE 0 but an expected test MSE of about 2. Explain both numbers.
(Hint: with distinct $x$ values, each leaf holds one observation. What does the tree predict at a
new $x$, and what is the variance of that prediction? Which method from lecture 1 does this
remind you of?)

<details>
<summary>Show answer: E4</summary>

**E4. Answer.** All 100 $x$ values are different, so a fully grown tree puts each observation in its own leaf and
predicts $y_i$ itself at $x_i$: the training MSE is 0. At a new $x$, it predicts the $y$ of the training
point whose leaf contains $x$, which is one of its neighbours. That is **1-nearest neighbour** (lecture 1).
That prediction is $f(x_i) + \varepsilon_i$ for an $x_i$ close to $x$, so its bias is close to 0 and its
variance is about $\sigma^2 = 1$. Adding the irreducible error gives about
$\sigma^2 + 0 + \sigma^2 = 2$. The table has $1 + 0.007 + 1.007 = 2.015$ (irreducible error + squared bias + variance).

</details>

---
## 3. Stopping rules

A **stopping rule** stops a tree from growing. sklearn has several. We try four, each over a range
of values, on the same 200 training sets:

- `max_depth`: at most this many splits from the root to any leaf;
- `min_samples_leaf`: every leaf must hold at least this many observations;
- `max_leaf_nodes`: grow the tree **best-first** (always making the split, anywhere in the tree,
  that reduces RSS the most) until it has this many leaves;
- `ccp_alpha`: grow the tree fully, then prune back every branch that doesn't reduce the RSS enough
  to pay for its leaves (cost-complexity pruning, Section 3.4).

For each, the table and the plot show the average number of leaves, the average training MSE, the
squared bias, the variance, and the expected test MSE. The plot shows expected test MSE minus
$\sigma^2 = 1$, which is ${bias}^2$ + variance, so that all four curves fit on one scale.

### 3.1 `max_depth`

In [ ]:
# Simulate different tree configurations
# Use our summarize function to compute bias^2, variance, and expected test MSE for different max_depth values
rows = [summarize(lambda X, y, d=d: DecisionTreeRegressor(max_depth=d).fit(X, y), f"max_depth = {d}")
        for d in [1, 2, 3, 4, 5, 6, 8, 10]]
rows.append(summarize(lambda X, y: DecisionTreeRegressor().fit(X, y), "fully grown"))
depth_df = pd.DataFrame(rows).set_index("setting")
display(depth_df.round(3))
plot_tradeoff(depth_df, "Stopping by max_depth")

**E5.** Which depth has the lowest expected test MSE? How do bias$^2$ and variance change as the depth
grows? Now look at the training MSE column. Would choosing the depth with the lowest training MSE be a
good idea?

<details>
<summary>Show answer: E5</summary>

**E5. Answer.** Depth 4 has the lowest expected test MSE, 1.415, with depth 3 close behind at 1.444. As the depth
grows, the squared bias falls fast, from 1.089 at depth 1 to 0.013 at depth 4, and then stays at about
0.007. The variance rises steadily, from 0.28 to 1.0. The training MSE falls all the way to 0, so choosing
the depth with the lowest training MSE would pick the fully grown tree, which has nearly the worst
expected test MSE (2.015). The training MSE is optimistic, and more so the bigger the tree: at depth 4 it is
0.684, against a test MSE of 1.415.

</details>

### 3.2 `min_samples_leaf`

Now the depth is unlimited, and the tree stops splitting a node when either child would have fewer
than `min_samples_leaf` observations. A **large** value means a **small** tree, so the table reads in
the opposite direction to the one above.

In [ ]:
# Use our summarize function to compute bias^2, variance, and expected test MSE for different min_samples_leaf values
# You can add n=1000 to the summarize function call, at the end, to see how the results change with a larger training set
rows = [summarize(lambda X, y, m=m: DecisionTreeRegressor(min_samples_leaf=m).fit(X, y), f"min_samples_leaf = {m}")
        for m in [1, 2, 3, 5, 8, 12, 20, 35]]
leaf_df = pd.DataFrame(rows).set_index("setting")
display(leaf_df.round(3))
plot_tradeoff(leaf_df, "Stopping by min_samples_leaf")

**E6.** Which value of `min_samples_leaf` is best, and how does its expected test MSE compare with the
best `max_depth`? `min_samples_leaf` averages at least that many $y$ values in every leaf. Use that
to explain why increasing it lowers the variance.

<details>
<summary>Show answer: E6</summary>

**E6. Answer.** `min_samples_leaf = 5` is best, with an expected test MSE of 1.425 and about 16 leaves. That is
essentially the same as the best depth (1.415). Each leaf predicts the mean of at least $m$ values of $y$,
and a mean of $m$ values has variance $\sigma^2/m$, so a bigger $m$ means less noisy predictions. You can see
this in the variance column: 1.007 at $m = 1$, 0.646 at $m = 2$, 0.379 at $m = 5$. It doesn't fall as fast
as $\sigma^2/m$ (which would be 0.2 at $m = 5$), because the cutpoints also move from one training set to
the next. Large leaves cost bias: at $m = 20$ the squared bias is 0.830.

</details>

**T1.** In the cell above, add the argument `n=1000` to `summarize` (so the line starts
`rows = [summarize(lambda X, y, m=m: ..., f"min_samples_leaf = {m}", n=1000)`), and rerun it. Which
value of `min_samples_leaf` is best now, and what happens to the ${bias}^2$ and variance of the best
tree? Remove `n=1000` afterwards and rerun the cell.

<details>
<summary>Show answer: T1</summary>

**T1. Answer.** With $n = 1000$, the best value is `min_samples_leaf = 20` (about 38 leaves), with squared bias
0.004, variance 0.100 and expected test MSE 1.104. With $n = 100$ the best was 5, with squared bias 0.046
and variance 0.379. With more data the tree can afford more leaves (less bias) *and* more observations in
each leaf (less variance), so both parts fall. The best setting changes with $n$: a fixed
`min_samples_leaf`, or a fixed depth, is not right for every sample size, so it has to be chosen from the
data. Notice also that a tree needs 1000 observations to do about as well as a smoothing spline does with
100 (see Q1).

</details>

### 3.3 `max_leaf_nodes`

In [ ]:
# Now use our summarize function to compute bias^2, variance, and expected test MSE for different max_leaf_nodes values
rows = [summarize(lambda X, y, k=k: DecisionTreeRegressor(max_leaf_nodes=k).fit(X, y), f"max_leaf_nodes = {k}")
        for k in [2, 4, 6, 8, 10, 12, 16, 24, 40]]
nodes_df = pd.DataFrame(rows).set_index("setting")
display(nodes_df.round(3))
plot_tradeoff(nodes_df, "Stopping by max_leaf_nodes")

**E7.** Compare the best `max_leaf_nodes` tree with the best `max_depth` tree. They can have about the
same number of leaves, but they put them in different places. Look back at Section 1: why might a
tree that is grown best-first use its leaves better than a tree that is grown level by level to a
fixed depth?

<details>
<summary>Show answer: E7</summary>

**E7. Answer.** The best `max_leaf_nodes` tree has 8 leaves and an expected test MSE of 1.401, the lowest so far.
The best `max_depth` tree needs about 15 leaves to reach 1.415. At the same size the difference is clearer:
`max_depth = 3` gives about 8 leaves and 1.444, against 1.401 for `max_leaf_nodes = 8`. A tree grown to a
fixed depth doesn't have the option of making the "best" splits. Best-first growth makes whichever split reduces RSS
most, anywhere in the tree, so it spends its leaves where $f$ changes fast, as in Section 1.

</details>

### 3.4 Cost-complexity pruning (`ccp_alpha`)

Cost-complexity pruning grows a full tree and then cuts it back: for a given $\alpha \ge 0$ it keeps
the subtree $T$ that minimizes
$$\mathrm{RSS}(T) + \alpha|T| \qquad\text{(ISL 8.4)},$$
where $|T|$ is the number of leaves. This formula should remind you of regularization for linear
models. Each leaf must buy at least $\alpha$ of RSS to be kept, so a **large** $\alpha$ means a
**small** tree. Here we use it like the other stopping rules: one fixed $\alpha$ for every training
set. (In practice $\alpha$ is chosen by cross-validation, ISL Algorithm 8.1.)

> **sklearn's `ccp_alpha` is ISL's $\alpha$ divided by $n$**, because sklearn uses the mean squared
> error instead of the RSS. The cell below uses ISL's $\alpha$ and passes `ccp_alpha = alpha / n`.


In [ ]:
# Prune a fully grown tree with the same alpha on every training set (ISL's alpha; sklearn's is alpha / n)
alpha_values = [0, 0.5, 1, 2, 3, 5, 8, 12, 20, 40, 80]
rows = [summarize(lambda X, y, a=a: DecisionTreeRegressor(ccp_alpha=a / len(y)).fit(X, y), f"alpha = {a}")
        for a in alpha_values]
prune_df = pd.DataFrame(rows).set_index("setting")
display(prune_df.round(3))
plot_tradeoff(prune_df, "Cost-complexity pruning")

**E8.** Which $\alpha$ is best, and how many leaves does it give on average? Compare its expected test MSE
with the best of the other three stopping rules.

<details>
<summary>Show answer: E8</summary>

**E8. Answer.** $\alpha = 5$ is best, with about 8.3 leaves on average and an expected test MSE of 1.425 (squared bias
0.027, variance 0.398). That is essentially a tie with the other three rules, whose best settings were
between 1.401 and 1.425. Pruning goes through the same trade-off as the other rules: a small $\alpha$
keeps a big tree with low bias and high variance, and a large one a small tree with high bias.

</details>


### 3.5 All four on one plot

The cell below puts the expected test MSE of every setting from this section on one plot, against the
average number of leaves. That compares the four ways of controlling a tree **at the same size**.


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for df, label, style in [(depth_df, "max_depth", "o--"), (leaf_df, "min_samples_leaf", "s-."),
                         (nodes_df, "max_leaf_nodes", "^:"), (prune_df, "pruning (alpha)", "D-")]:
    ax.plot(df["mean leaves"], df["expected test MSE"], style, lw=2 if label.startswith("pruning") else 1.2, label=label)
ax.set(xscale="log", xlabel="mean number of leaves", ylabel="expected test MSE",
       ylim=(1.3, 2.5), title="Four ways to control the size of a tree")
ax.legend()
plt.show()

best = pd.DataFrame({name: df["expected test MSE"].agg(["idxmin", "min"])
                     for name, df in [("max_depth", depth_df), ("min_samples_leaf", leaf_df),
                                      ("max_leaf_nodes", nodes_df), ("pruning", prune_df)]}).T
display(best)

**E9.** Which method is best at its best setting, and which is worst? Are the differences large compared
with the difference between a good and a bad *size*? What does that suggest you should spend your
effort on when fitting a single tree?

<details>
<summary>Show answer: E9</summary>

**E9. Answer.** At their best settings, `max_leaf_nodes` is best (1.401) and `min_samples_leaf` and pruning are worst
(both 1.425), with `max_depth` in between (1.415). The differences are at most 0.025. The difference
between a good size and a bad one is far bigger: the same curves run from about 1.4 at 8 to 16 leaves up to
2.0 for a fully grown tree and 2.4 for a stump. So for a single tree, what matters is choosing the **size**
well, by cross-validation, and much less which rule you use to get there. The plot shows one more thing: at
sizes above about 10 leaves, pruning a fully grown tree does worse than `min_samples_leaf` at the same size,
because pruning can keep leaves with only one or two observations, whose means are very noisy. Growing
the tree with `min_samples_leaf=5` before pruning it, as ISL and ESL suggest, fixes that.

</details>

---
## 4. Wrap-up questions

**Q1.** On this same problem ($n = 100$, $\sigma = 1$), a smoothing spline with 10 degrees of freedom,
as in the splines lab, has an expected test MSE of about 1.10 (from a simulation like the ones in
this notebook). The best trees here are at about 1.40. Look at the ${bias}^2$ and variance of the best trees: which is the main
reason? A least squares fit on 10 *fixed* basis functions would have a variance of about
$\sigma^2 \times 10/n = 0.1$. Compare that with the variance of the 10-leaf tree in Section 3.3, and
explain the difference.

<details>
<summary>Show answer: Q1</summary>

**Q1. Answer.** Variance. The best trees have a small squared bias (0.016 for the 10-leaf tree in Section 3.3) but a
variance of about 0.40 (0.403 for that tree). That is four times the 0.1 of a least squares fit on 10 fixed
basis functions. The difference is that a tree's cuts are not fixed: they are chosen by searching for the
splits that best fit *this* training set, noise included. Each training set puts the cuts in different places,
and the leaf means are fitted to the noise that the search selected. So the tree behaves like a model with far
more than 10 parameters. The squared bias is small because,
averaged over training sets, the steps fall in different places and the average tree is smooth. So the cost
of the step shape shows up as **variance**, not bias. However it is grown or pruned, a 10-leaf tree has to
choose its cutpoints from the data, and that is where the variance comes from.

</details>

**Q2.** Next lecture's **bagging** fits a fully grown tree to each of many bootstrap samples and averages
their predictions. Using Section 2, which part of the fully grown tree's expected test MSE would
averaging reduce, and which would it leave alone? Why grow the trees fully, rather than prune them?

<details>
<summary>Show answer: Q2</summary>

**Q2. Answer.** Averaging would reduce the **variance**, which is almost all of the fully grown tree's error: 1.007 of
the 2.015, against a squared bias of 0.007. It would leave the bias roughly alone, because the average of many
trees has about the same expected value as one tree. The trees are grown fully because fully grown trees have
the lowest bias. Averaging can remove variance but not bias, so you want each tree to have as little bias as
possible and let the averaging deal with the variance. Pruning each tree would add bias that averaging can't
take away. (Bootstrap samples overlap, so the trees are correlated and the variance doesn't fall all the way to
$1.007/B$. Random forests, next lecture, reduce that correlation.)

</details>

**Q3.** A colleague chooses `max_depth` by fitting trees of depth 1 to 10 and picking the one with the
lowest training MSE. What will they pick, and what is wrong with it? What should they do instead?

<details>
<summary>Show answer: Q3</summary>

**Q3. Answer.** They will pick depth 10, the deepest. The training MSE always falls as the tree grows (from 0.684 at depth
4 to 0.030 at depth 10 in Section 3.1), because each split reduces the training RSS. But depth 10 has an
expected test MSE of 1.988, against 1.415 at depth 4. Training error is optimistic, and more so for bigger
trees, so it can't be used to choose the size. Choose the depth (or $\alpha$) by cross-validation
(for example with `GridSearchCV`), or by a validation set.

</details>